<a href="https://colab.research.google.com/github/shims79757-lang/Elevance-Skills-Projects/blob/main/App_Category_Performance_Streamgraph_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==========================================
# 1. Imports & Setup
# ==========================================
from datetime import datetime
from zoneinfo import ZoneInfo
import numpy as np
import pandas as pd
import plotly.graph_objects as go

# ==========================================
# 2. Load Dataset
# ==========================================
url = "https://raw.githubusercontent.com/shims79757-lang/Elevance-Skills-Projects/main/googleplaystore.csv"
apps = pd.read_csv(url)
data = apps.copy()

# ==========================================
# 3. Data Cleaning & Type Conversion
# ==========================================
data["Rating"] = pd.to_numeric(data["Rating"], errors="coerce")
data["Reviews"] = pd.to_numeric(data["Reviews"], errors="coerce")

# Clean Installs (remove '+' and ',')
data["Installs"] = (
    data["Installs"]
    .astype(str)
    .str.replace(",", "", regex=False)
    .str.replace("+", "", regex=False)
)
data["Installs"] = pd.to_numeric(data["Installs"], errors="coerce")


# Convert Size string to megabytes (MB)
def convert_size(size):
  size = str(size).strip()
  if size.endswith("M"):
    return float(size[:-1])
  elif size.endswith("k"):
    return float(size[:-1]) / 1024
  else:
    return np.nan


data["Size_MB"] = data["Size"].apply(convert_size)

# Parse date and drop invalid records
data["Last Updated"] = pd.to_datetime(data["Last Updated"], errors="coerce")
data = data.dropna(
    subset=["Last Updated", "Rating", "Installs", "Reviews", "Size_MB"]
)

# ==========================================
# 4. Category Filtering & Translations
# ==========================================
# Filter categories beginning with 'T', 'P', or 'B'
data = data[
    data["Category"].str.upper().str.startswith(("T", "P", "B"))
].copy()


def translate_category(cat):
  cat_upper = str(cat).strip().upper()
  if cat_upper in ["TRAVEL_AND_LOCAL", "TRAVEL & LOCAL", "TRAVEL AND LOCAL"]:
    return "Voyages et local"  # French
  elif cat_upper == "PRODUCTIVITY":
    return "Productividad"  # Spanish
  elif cat_upper == "PHOTOGRAPHY":
    return "写真"  # Japanese
  else:
    return str(cat).replace("_", " ").title()


data["Display_Category"] = data["Category"].apply(translate_category)

# ==========================================
# 5. App-Level Filtering Criteria
# ==========================================
# - Rating >= 4.2
# - Reviews > 1,000
# - Size between 20 MB and 80 MB
# - Installs >= 10,000
# - App names containing NO numbers
data_filtered = data[
    (data["Rating"] >= 4.2)
    & (data["Reviews"] > 1000)
    & (data["Size_MB"] >= 20.0)
    & (data["Size_MB"] <= 80.0)
    & (data["Installs"] >= 10000)
    & (~data["App"].str.contains(r"\d", regex=True, na=False))
].copy()

data_filtered = data_filtered.drop_duplicates(
    subset=["App", "Display_Category"]
).reset_index(drop=True)

# ==========================================
# 6. Monthly Timeline & Missing Month Imputation
# ==========================================
data_filtered["YearMonth"] = (
    data_filtered["Last Updated"].dt.to_period("M").dt.to_timestamp()
)

# Aggregate monthly installs
monthly_installs = (
    data_filtered.groupby(["YearMonth", "Display_Category"])["Installs"]
    .sum()
    .unstack(fill_value=0)
)

# Create continuous timeline and fill missing months with 0
if not monthly_installs.empty:
  full_timeline = pd.date_range(
      start=monthly_installs.index.min(),
      end=monthly_installs.index.max(),
      freq="MS",
  )
  monthly_installs = monthly_installs.reindex(full_timeline, fill_value=0)

# ==========================================
# 7. Cumulative Installs & MoM Growth
# ==========================================
cumulative_installs = monthly_installs.cumsum()

mom_growth = pd.DataFrame(
    index=monthly_installs.index, columns=monthly_installs.columns
)
for col in monthly_installs.columns:
  prev = monthly_installs[col].shift(1)
  curr = monthly_installs[col]
  growth = np.where(
      prev > 0,
      ((curr - prev) / prev) * 100.0,
      np.where(curr > 0, 100.0, 0.0),
  )
  mom_growth[col] = growth

# ==========================================
# 8. Anomaly Detection (MoM > 25% & Rolling Z-Score > 2)
# ==========================================
anomalies = []
z_scores = pd.DataFrame(
    index=monthly_installs.index, columns=monthly_installs.columns
)

for col in monthly_installs.columns:
  # Rolling 3-month window
  roll_mean = mom_growth[col].rolling(window=3, min_periods=2).mean()
  roll_std = (
      mom_growth[col]
      .rolling(window=3, min_periods=2)
      .std()
      .replace(0, np.nan)
  )

  z = (mom_growth[col] - roll_mean) / roll_std
  z = z.fillna(0)
  z_scores[col] = z

  anomaly_mask = (mom_growth[col] > 25.0) & (z > 2.0)
  for date, is_anom in anomaly_mask.items():
    if is_anom:
      anomalies.append({
          "category": col,
          "date": date,
          "monthly_installs": monthly_installs.loc[date, col],
          "cumulative_installs": cumulative_installs.loc[date, col],
          "growth": mom_growth.loc[date, col],
          "zscore": z_scores.loc[date, col],
      })

# ==========================================
# 9. Build Interactive Streamgraph in Plotly
# ==========================================
categories_list = list(monthly_installs.columns)
palette = [
    "#4361EE",
    "#3A0CA3",
    "#7209B7",
    "#F72585",
    "#4CC9F0",
    "#06D6A0",
    "#FFB703",
    "#FB8500",
    "#E63946",
    "#118AB2",
]
color_map = {
    cat: palette[i % len(palette)] for i, cat in enumerate(categories_list)
}

fig = go.Figure()

# Trace Group 1: Monthly Installs (Stacked Streamgraph)
for cat in categories_list:
  fig.add_trace(
      go.Scatter(
          x=monthly_installs.index,
          y=monthly_installs[cat],
          name=cat,
          mode="lines",
          stackgroup="monthly_stream",
          line=dict(width=0.8, color=color_map[cat]),
          fillcolor=color_map[cat],
          opacity=0.75,
          hovertemplate=(
              f"<b>{cat}</b><br>Month: %{{x|%b %Y}}<br>Monthly Installs:"
              " %{y:,.0f}<extra></extra>"
          ),
          visible=True,
      )
  )

# Trace Group 2: Cumulative Installs (Stacked Area)
for cat in categories_list:
  fig.add_trace(
      go.Scatter(
          x=cumulative_installs.index,
          y=cumulative_installs[cat],
          name=cat,
          mode="lines",
          stackgroup="cumulative_stream",
          line=dict(width=0.8, color=color_map[cat]),
          fillcolor=color_map[cat],
          opacity=0.75,
          hovertemplate=(
              f"<b>{cat}</b><br>Month: %{{x|%b %Y}}<br>Cumulative Installs:"
              " %{y:,.0f}<extra></extra>"
          ),
          visible=False,
      )
  )

# Trace Group 3: Growth Percentage (Line Chart)
for cat in categories_list:
  fig.add_trace(
      go.Scatter(
          x=mom_growth.index,
          y=mom_growth[cat],
          name=cat,
          mode="lines+markers",
          line=dict(width=2, color=color_map[cat]),
          marker=dict(size=5),
          opacity=0.75,
          hovertemplate=(
              f"<b>{cat}</b><br>Month: %{{x|%b %Y}}<br>MoM Growth:"
              " %{y:.1f}%<extra></extra>"
          ),
          visible=False,
      )
  )

# Anomaly Overlays (Higher Intensity / Diamond Markers)
num_cats = len(categories_list)
has_anoms = len(anomalies) > 0

if has_anoms:
  anom_x = [a["date"] for a in anomalies]
  anom_text = [
      f"<b>{a['category']}</b><br>Growth: +{a['growth']:.1f}%<br>Z-Score:"
      f" {a['zscore']:.2f}"
      for a in anomalies
  ]

  # Monthly anomaly trace
  fig.add_trace(
      go.Scatter(
          x=anom_x,
          y=[a["monthly_installs"] for a in anomalies],
          mode="markers",
          name="Anomalies (Growth > 25%, Z > 2)",
          marker=dict(
              symbol="diamond",
              size=13,
              color="#FF0055",
              line=dict(color="#FFFFFF", width=2),
          ),
          text=anom_text,
          hoverinfo="text",
          visible=True,
      )
  )

  # Cumulative anomaly trace
  fig.add_trace(
      go.Scatter(
          x=anom_x,
          y=[a["cumulative_installs"] for a in anomalies],
          mode="markers",
          name="Anomalies (Cumulative View)",
          marker=dict(
              symbol="diamond",
              size=13,
              color="#FF0055",
              line=dict(color="#FFFFFF", width=2),
          ),
          text=anom_text,
          hoverinfo="text",
          visible=False,
      )
  )

  # Growth anomaly trace
  fig.add_trace(
      go.Scatter(
          x=anom_x,
          y=[a["growth"] for a in anomalies],
          mode="markers",
          name="Anomalies (Growth View)",
          marker=dict(
              symbol="diamond",
              size=13,
              color="#FF0055",
              line=dict(color="#FFFFFF", width=2),
          ),
          text=anom_text,
          hoverinfo="text",
          visible=False,
      )
  )

# Setup trace visibility vectors for button toggles
monthly_vis = (
    [True] * num_cats
    + [False] * num_cats
    + [False] * num_cats
    + ([True, False, False] if has_anoms else [])
)
cum_vis = (
    [False] * num_cats
    + [True] * num_cats
    + [False] * num_cats
    + ([False, True, False] if has_anoms else [])
)
growth_vis = (
    [False] * num_cats
    + [False] * num_cats
    + [True] * num_cats
    + ([False, False, True] if has_anoms else [])
)

# Annotations for anomaly callouts
annotations = []
for a in anomalies:
  annotations.append(
      dict(
          x=a["date"],
          y=a["monthly_installs"],
          text=f"<b>+{a['growth']:.0f}%</b> (Z={a['zscore']:.1f})",
          showarrow=True,
          arrowhead=2,
          arrowsize=1,
          arrowwidth=1.5,
          arrowcolor="#FF0055",
          ax=0,
          ay=-35,
          bgcolor="rgba(255, 255, 255, 0.9)",
          bordercolor="#FF0055",
          borderwidth=1,
          font=dict(size=10, color="#FF0055"),
      )
  )

# Layout & Interactive Controls
fig.update_layout(
    updatemenus=[
        dict(
            type="buttons",
            direction="left",
            active=0,
            x=0.0,
            y=1.18,
            xanchor="left",
            yanchor="top",
            buttons=[
                dict(
                    label="📊 Monthly Installs",
                    method="update",
                    args=[
                        {"visible": monthly_vis},
                        {
                            "yaxis": {
                                "title": "Monthly Installs",
                                "tickformat": ",.0f",
                            },
                            "title": (
                                "<b>Monthly App Installs by Category</b><br><sup>Streamgraph"
                                " with Anomalous Periods Highlighted (Growth >"
                                " 25%, Z > 2)</sup>"
                            ),
                            "annotations": annotations,
                        },
                    ],
                ),
                dict(
                    label="📈 Cumulative Installs",
                    method="update",
                    args=[
                        {"visible": cum_vis},
                        {
                            "yaxis": {
                                "title": "Cumulative Installs",
                                "tickformat": ",.0f",
                            },
                            "title": (
                                "<b>Cumulative App Installs Over Time</b><br><sup>Total"
                                " aggregate installs across eligible"
                                " categories</sup>"
                            ),
                            "annotations": [],
                        },
                    ],
                ),
                dict(
                    label="⚡ Growth Percentage (%)",
                    method="update",
                    args=[
                        {"visible": growth_vis},
                        {
                            "yaxis": {
                                "title": "Month-over-Month Growth (%)",
                                "ticksuffix": "%",
                            },
                            "title": (
                                "<b>Month-over-Month Growth Rate"
                                " (%)</b><br><sup>Monitoring trajectory and"
                                " anomaly thresholds</sup>"
                            ),
                            "annotations": [],
                        },
                    ],
                ),
            ],
        )
    ],
    title=dict(
        text=(
            "<b>Monthly App Installs by Category</b><br><sup>Streamgraph with"
            " Anomalous Periods Highlighted (Growth > 25%, Z > 2)</sup>"
        ),
        x=0.5,
        xanchor="center",
    ),
    xaxis=dict(
        title="Timeline (Month-Year)",
        type="date",
        tickformat="%b %Y",
        showgrid=True,
        gridcolor="#E5E7EB",
    ),
    yaxis=dict(
        title="Monthly Installs",
        tickformat=",.0f",
        showgrid=True,
        gridcolor="#E5E7EB",
    ),
    template="plotly_white",
    height=750,
    hovermode="x unified",
    margin=dict(l=70, r=40, t=130, b=70),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    annotations=annotations,
)

# ==========================================
# 10. Time-Gated Display Restriction (4 PM - 6 PM IST)
# ==========================================
current_time = datetime.now(ZoneInfo("Asia/Kolkata"))
print(
    "Current Local Time (IST):",
    current_time.strftime("%d %B %Y, %I:%M:%S %p"),
)

# Enforce window: 16:00 to 18:00 IST
if 16 <= current_time.hour < 18:
  print("✅ Active Window (4:00 PM - 6:00 PM IST): Displaying visualization.")
  fig.show()
else:
  print("🚫 Visualization unavailable.")
  print(
      "This visualization is scheduled to appear on the dashboard only between"
      " 4:00 PM and 6:00 PM IST."
  )